# GNSS daily means and interpolation dates

This notebook reads processed `.pos` files for the 29 paper stations from 2011–2026. It calculates daily GNSS position and height means, then selects 90-day interpolation-window centers at 30-day steps when a daily solution falls within 16 days of the center. Coordinates are projected to Antarctic Polar Stereographic (EPSG:3031).

Input layout: `refl_code/<year>/pos/<station>/*.pos`. The notebook writes an all-station daily-mean CSV, per-station daily-mean CSVs, and the interpolation mid-date CSV used by the REMA variogram script.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from pyproj import Transformer
except ImportError:  # Allows daily lat/lon output even if pyproj is unavailable.
    Transformer = None


# ============================================================
# CONFIG
# ============================================================

# Set ROOT to the DockerPreprocessing output containing <year>/pos/<station>/*.pos.
ROOT = Path("/Volumes/cornice/marquardt/data/GNSS/docker/local/refl_code")
# OUT_DIR receives the daily means and interpolation-date CSVs used by the REMA fit.
OUT_DIR = Path("/Volumes/cornice/marquardt/results/gnss_daily_means_from_pos")

PAPER_STATIONS = [
    "gz01",
    "gz02",
    "gz06",
    "gz09",
    "gz13",
    "gz17",
    "gz18",
    "la01",
    "la02",
    "la03",
    "la04",
    "la05",
    "la06",
    "la07",
    "la08",
    "la09",
    "la10",
    "la11",
    "la12",
    "la13",
    "la14",
    "la15",
    "la16",
    "la17",
    "la18",
    "mg04",
    "mg05",
    "mg06",
    "mg07",
]
YEAR_START, YEAR_END = 2011, 2026

DAILY_ALL_CSV = OUT_DIR / "gnss_daily_means_2011_2026_29_paper_stations.csv"
INTERP_CSV = OUT_DIR / "gnss_interpolation_mid_dates_2011_2026_29_paper_stations.csv"

WINDOW_DAYS = 90
HALF_WINDOW = WINDOW_DAYS // 2
STEP_DAYS = 30
MAX_MATCH_DAYS = 16

# Antarctic Polar Stereographic. Change if your x/y target CRS is different.
GNSS_XY_EPSG = "EPSG:3031"


# ============================================================
# POS PARSING HELPERS
# ============================================================

COMMON_POS_NAMES = [
    "date",
    "time",
    "latitude",
    "longitude",
    "height",
    "Q",
    "ns",
    "sdn",
    "sde",
    "sdu",
    "sdne",
    "sdeu",
    "sdun",
    "age",
    "ratio",
]


def find_pos_files(root: Path) -> list[Path]:
    """Find paper-station POS files for the configured years."""
    files: list[Path] = []

    for year in range(YEAR_START, YEAR_END + 1):
        pos_dir = root / str(year) / "pos"
        for station in PAPER_STATIONS:
            files.extend(sorted((pos_dir / station).glob("*.pos")))

    return files


def station_from_path(pos_path: Path) -> str:
    """For ROOT/{year}/pos/{station}/file.pos, station is the parent directory."""
    return pos_path.parent.name


def date_from_path(pos_path: Path) -> pd.Timestamp | pd.NaT:
    """
    Fallback date parser for POS files without epoch columns.

    Handles common filename patterns containing YYYYMMDD, YYYY-DDD, YYYYDDD,
    or station + DOY when the year is present in the directory path.
    """
    text = str(pos_path)
    name = pos_path.stem

    match = re.search(r"(20\d{2})(\d{2})(\d{2})", name)
    if match:
        return pd.Timestamp(
            year=int(match.group(1)),
            month=int(match.group(2)),
            day=int(match.group(3)),
        )

    match = re.search(r"(20\d{2})[-_]?([0-3]\d{2})", name)
    if match:
        return pd.Timestamp(year=int(match.group(1)), month=1, day=1) + pd.Timedelta(
            days=int(match.group(2)) - 1
        )

    year_match = re.search(r"/(20\d{2})/pos/", text)
    doy_match = re.search(r"(?<!\d)([0-3]\d{2})(?!\d)", name)
    if year_match and doy_match:
        return pd.Timestamp(year=int(year_match.group(1)), month=1, day=1) + pd.Timedelta(
            days=int(doy_match.group(1)) - 1
        )

    return pd.NaT


def read_pos(pos_path: Path) -> pd.DataFrame:
    """
    Read an RTKLIB-style .pos file.

    Expected common data rows:
        YYYY/MM/DD HH:MM:SS latitude longitude height ...

    Header/comment lines and variable-width non-data rows are ignored.
    """
    rows: list[dict[str, object]] = []

    with pos_path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("%") or line.startswith("#"):
                continue

            parts = line.split()
            if len(parts) < 5:
                continue

            # RTKLIB .pos rows usually start with date/time. If this is a
            # header or other metadata row, the datetime conversion below will
            # fail and the row will be dropped.
            rows.append(
                {
                    "date": parts[0],
                    "time": parts[1],
                    "latitude": parts[2],
                    "longitude": parts[3],
                    "height": parts[4],
                }
            )

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    for col in ["latitude", "longitude", "height"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if {"date", "time"}.issubset(df.columns):
        df["datetime"] = pd.to_datetime(
            df["date"].astype(str) + " " + df["time"].astype(str),
            errors="coerce",
        )
    else:
        df["datetime"] = pd.NaT

    if df["datetime"].isna().all():
        fallback_date = date_from_path(pos_path)
        if pd.notna(fallback_date):
            df["datetime"] = fallback_date

    return df.dropna(subset=["latitude", "longitude", "datetime"])


# ============================================================
# DAILY MEANS
# ============================================================


def add_projected_xy(daily: pd.DataFrame) -> pd.DataFrame:
    if daily.empty:
        daily["x_gnss"] = pd.Series(dtype=float)
        daily["y_gnss"] = pd.Series(dtype=float)
        return daily

    if Transformer is None:
        daily["x_gnss"] = np.nan
        daily["y_gnss"] = np.nan
        print("WARNING: pyproj is not installed, so x_gnss/y_gnss were set to NaN.")
        return daily

    transformer = Transformer.from_crs("EPSG:4326", GNSS_XY_EPSG, always_xy=True)
    x, y = transformer.transform(daily["lon_mean"].to_numpy(), daily["lat_mean"].to_numpy())
    daily["x_gnss"] = x
    daily["y_gnss"] = y
    return daily


def build_daily_means(pos_files: list[Path]) -> pd.DataFrame:
    rows: list[dict[str, object]] = []

    for i, pos_path in enumerate(pos_files, start=1):
        if i % 500 == 0:
            print(f"Read {i:,}/{len(pos_files):,} POS files...")

        station = station_from_path(pos_path)
        df = read_pos(pos_path)
        if df.empty:
            print(f"[SKIP_EMPTY_OR_UNREADABLE] {pos_path}")
            continue

        df["mid_day"] = df["datetime"].dt.normalize()

        grouped = (
            df.groupby("mid_day", as_index=False)
            .agg(
                lat_mean=("latitude", "mean"),
                lon_mean=("longitude", "mean"),
                height_mean=("height", "mean"),
                n_epochs=("latitude", "size"),
            )
            .assign(station=station, source_file=str(pos_path))
        )

        rows.extend(grouped.to_dict("records"))

    daily = pd.DataFrame(rows)
    if daily.empty:
        return daily

    for col in ["lat_mean", "lon_mean", "height_mean"]:
        daily[f"{col}_weighted"] = daily[col] * daily["n_epochs"]

    daily = (
        daily.groupby(["station", "mid_day"], as_index=False)
        .agg(
            lat_mean_weighted=("lat_mean_weighted", "sum"),
            lon_mean_weighted=("lon_mean_weighted", "sum"),
            height_mean_weighted=("height_mean_weighted", "sum"),
            n_epochs=("n_epochs", "sum"),
            n_pos_files=("source_file", "nunique"),
        )
        .sort_values(["station", "mid_day"])
        .reset_index(drop=True)
    )
    daily["lat_mean"] = daily["lat_mean_weighted"] / daily["n_epochs"]
    daily["lon_mean"] = daily["lon_mean_weighted"] / daily["n_epochs"]
    daily["height_mean"] = daily["height_mean_weighted"] / daily["n_epochs"]

    daily = add_projected_xy(daily)
    return daily[
        [
            "station",
            "mid_day",
            "lat_mean",
            "lon_mean",
            "height_mean",
            "x_gnss",
            "y_gnss",
            "n_epochs",
            "n_pos_files",
        ]
    ]


def write_station_csvs(daily: pd.DataFrame, out_dir: Path) -> None:
    station_dir = out_dir / "station_daily_means_2011_2026_29_paper_stations"
    station_dir.mkdir(parents=True, exist_ok=True)

    for station, g in daily.groupby("station", sort=True):
        safe_station = re.sub(r"[^A-Za-z0-9_.-]+", "_", station)
        out_csv = station_dir / f"{safe_station}_daily_mean_location_2011_2026.csv"
        g.sort_values("mid_day").to_csv(out_csv, index=False)


# ============================================================
# INTERPOLATION MID DATES
# ============================================================


def build_interpolation_mid_dates(daily: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, object]] = []

    for st, g in daily.groupby("station", sort=False):
        g = g.sort_values("mid_day").reset_index(drop=True)
        if g.empty:
            continue

        first_day = g["mid_day"].min()
        last_day = g["mid_day"].max()

        center = first_day + pd.Timedelta(days=HALF_WINDOW)

        while center <= last_day - pd.Timedelta(days=HALF_WINDOW):
            idx = (g["mid_day"] - center).abs().idxmin()
            matched_row = g.loc[idx]
            day_diff = int((matched_row["mid_day"] - center).days)

            rows.append(
                {
                    "station": st,
                    "mid_dt": center,
                    "mid_day": center.normalize(),
                    "matched_date": matched_row["mid_day"],
                    "day_diff": day_diff,
                    "lat_mean": matched_row["lat_mean"],
                    "lon_mean": matched_row["lon_mean"],
                    "height_mean": matched_row["height_mean"],
                    "x_gnss": matched_row["x_gnss"],
                    "y_gnss": matched_row["y_gnss"],
                    "n_epochs": matched_row["n_epochs"],
                    "n_pos_files": matched_row["n_pos_files"],
                }
            )

            center += pd.Timedelta(days=STEP_DAYS)

    interp = pd.DataFrame(rows)
    if interp.empty:
        return interp

    return (
        interp[np.abs(interp["day_diff"]) < MAX_MATCH_DAYS]
        .copy()
        .sort_values(["station", "mid_day"])
        .reset_index(drop=True)
    )


def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Searching POS files under: {ROOT}")
    pos_files = find_pos_files(ROOT)
    print(f"Found POS files: {len(pos_files):,}")

    if not pos_files:
        raise FileNotFoundError(f"No .pos files found under {ROOT}/{{year}}/pos/{{station}}")

    daily = build_daily_means(pos_files)
    if daily.empty:
        raise RuntimeError("No valid daily means were produced from the POS files.")

    daily.to_csv(DAILY_ALL_CSV, index=False)
    write_station_csvs(daily, OUT_DIR)

    interp_good = build_interpolation_mid_dates(daily)
    interp_good.to_csv(INTERP_CSV, index=False)

    print("")
    print("Done.")
    print(f"Daily all-station CSV: {DAILY_ALL_CSV}")
    print(f"Per-station CSV folder: {OUT_DIR / 'station_daily_means_2011_2026_29_paper_stations'}")
    print(f"Interpolation mid-date CSV: {INTERP_CSV}")
    print(f"Daily rows: {len(daily):,}")
    print(f"Stations: {daily['station'].nunique():,}")
    print(f"Good interpolation windows: {len(interp_good):,}")


if __name__ == "__main__":
    main()
